# Day 17: Project 1 — API + Docker Deployment

**Model:** Best LightGBM (from Day 14)
**Goal:** FastAPI `/predict` endpoint with Pydantic validation, Dockerize, test locally

In [ ]:
import subprocess
import sys

# Build Docker image
print("Building Docker image...")
result = subprocess.run(["docker", "build", "-t", "readmission-api", "."], 
                       cwd="../projects/project1_readmission", 
                       capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print(f"Return code: {result.returncode}")

In [ ]:
# Test the API (run container in background, test, then stop)
import subprocess
import time
import requests
import json

print("Starting container...")
container = subprocess.Popen([
    "docker", "run", "-d", "-p", "8001:8000", 
    "--name", "readmission-test",
    "readmission-api"
], cwd="../projects/project1_readmission")

container_id = container.stdout.read().decode().strip()
print(f"Container started: {container_id}")

# Wait for startup
time.sleep(5)

# Test health endpoint
try:
    resp = requests.get("http://localhost:8001/health")
    print(f"Health check: {resp.status_code} - {resp.json()}")
except Exception as e:
    print(f"Health check failed: {e}")

# Test prediction
test_patient = {
    "race": "Caucasian",
    "gender": "Female",
    "age": "[70-80)",
    "weight": "[75-100)",
    "admission_type_id": 1,
    "discharge_disposition_id": 1,
    "admission_source_id": 7,
    "time_in_hospital": 4,
    "payer_code": "MC",
    "medical_specialty": "Cardiology",
    "num_lab_procedures": 45,
    "num_procedures": 1,
    "num_medications": 12,
    "number_outpatient": 0,
    "number_emergency": 0,
    "number_inpatient": 0,
    "number_diagnoses": 7,
    "metformin": "No",
    "repaglinide": "No",
    "nateglinide": "No",
    "chlorpropamide": "No",
    "glimepiride": "No",
    "acetohexamide": "No",
    "glipizide": "No",
    "glyburide": "No",
    "tolbutamide": "No",
    "pioglitazone": "No",
    "rosiglitazone": "No",
    "acarbose": "No",
    "miglitol": "No",
    "troglitazone": "No",
    "tolazamide": "No",
    "examide": "No",
    "citoglipton": "No",
    "insulin": "No",
    "glyburide-metformin": "No",
    "glipizide-metformin": "No",
    "glimepiride-pioglitazone": "No",
    "metformin-rosiglitazone": "No",
    "metformin-pioglitazone": "No",
    "change": "No",
    "diabetesMed": "No",
    "diag_1_cat": "Circulatory",
    "diag_2_cat": "Endocrine",
    "diag_3_cat": "Respiratory",
    "prior_admissions": 0,
    "med_change_count": 0,
    "los_category": "Medium (4-7d)",
    "num_diagnoses_cat": "Moderate (6-8)",
    "age_midpoint": 75
}

try:
    resp = requests.post(
        "http://localhost:8001/predict",
        json=test_patient,
        headers={"Content-Type": "application/json"}
    )
    print(f"Prediction: {resp.status_code}")
    print(json.dumps(resp.json(), indent=2))
except Exception as e:
    print(f"Prediction failed: {e}")

# Cleanup
subprocess.run(["docker", "stop", container_id])
subprocess.run(["docker", "rm", container_id])
print("Container stopped and removed")

## Summary Notes

- Docker build: 
- API health check: 
- Prediction test: 
- Response format: 